# Многомерная LSTM

На каждом шаге LSTM получает все N рядов. Последнее скрытое состояние преобразуется в N прогнозов. Граф не используется.

Каждый запуск этого ноутбука обрабатывает **все три набора**: синтетику, MOEX и METR-LA.
Наборы загружаются по очереди. Вспомогательные функции находятся в `src/forecast_*.py`.
Результаты каждого опыта и сводный CSV сохраняются автоматически.

Нужны `numpy`, `pandas`, `torch`. Отдельные графовые библиотеки не требуются.

По умолчанию: один seed, CPU, до 15 эпох с ранней остановкой для каждого набора.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from forecast_batch import run_batch
from forecast_experiment import collect_results, fixed_graph
from forecast_models import build_adjacency

print("Проект:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)


Проект: /home/loktar/Desktop/money_coursework
PyTorch: 2.15.0+cpu


## Настройки серии

Каждый ноутбук запускает свою модель на `synthetic`, `moex`, `metr_la` по очереди.
История и горизонт читаются из `schema.json`; текущая постановка — 60 наблюдений и один шаг вперёд.
Используйте одинаковые `DATASETS`, `CONFIG`, `SEEDS`, `RUN_TAG` во всех четырёх ноутбуках.
Один seed — пилот; для оценки разброса позже задайте `[0, 1, 2]` во всех четырёх.

`EVALUATE_TEST=True`: после выбора лучшей эпохи по val рассчитывается test.
Test не участвует в обучении и выборе эпохи. Для новой настройки по val можно задать `False`.
Уже просмотренный test учитываем как исследовательскую оценку при дальнейшей настройке.


In [2]:
MODEL_KIND = "lstm"
DATASETS = ["synthetic", "moex", "metr_la"]
SEEDS = [0]
RUN_TAG = "pilot_v1"
EVALUATE_TEST = True

CONFIG = {
    "hidden_size": 16,
    "batch_size": 32,
    "max_epochs": 15,
    "patience": 4,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "gradient_clip": 1.0,
    "top_k": 3,
    "cpu_threads": 2,
    "device": "cpu",
    "min_delta": 0.0,
}

DATA_ROOT = PROJECT_ROOT / "data/processed"
RESULTS_ROOT = PROJECT_ROOT / "results"
print("Модель:", MODEL_KIND)
print("Наборы:", DATASETS)


Модель: lstm
Наборы: ['synthetic', 'moex', 'metr_la']


## Общий загрузчик и покрытие целей

CSV каждого набора читаются один раз при его запуске. Масштабирование каждого ряда обучается только по наблюдаемым train-значениям.
Пропуски входа заполняются прошлым значением только внутри выбранного окна, затем обучающим средним.
Цели остаются пропущенными; `y_mask` применяется и в функции потерь, и во всех метриках.

Таблица ниже показывает **фактическое число оцениваемых целей** каждой вершины.
Вершина с нулевым покрытием остаётся в составе, но не получает выдуманную нулевую ошибку.
Состав акций по доступности test не пересматривается.

Функция ниже вызывается автоматически внутри цикла по наборам.


In [3]:
def inspect_dataset(data, dataset_name):
    # Keep inspection local so the previous dataset can be released.
    display(pd.DataFrame([
        {"dataset": dataset_name, "split": split, "samples": len(data.dataset(split)),
         "nodes": len(data.nodes), "history": data.history, "features_per_node": 1}
        for split in ("train", "val", "test")
    ]))
    coverage = data.quality().pivot(index="node", columns="split", values="evaluated_targets")
    display(coverage.head(12))
    zero_coverage = coverage.loc[coverage.eq(0).any(axis=1)]
    if len(zero_coverage):
        print("Вершины без оцениваемых целей хотя бы в одном разбиении:")
        display(zero_coverage)
    example = data.dataset("val")[0]
    print("Один вход:", example["X"].shape, "цель:", example["y_norm"].shape)


## Запуск на трёх наборах

Нейросети обучаются по MSE нормированных целей с маской. Лучший checkpoint выбирается только по val:
по MAE для синтетики и METR-LA, по QLIKE для MOEX. У всех нейросетей одинаковые правила оптимизации,
ранней остановки и перемешивания обучающих примеров.

MAE/RMSE считаются в исходной шкале. Для MOEX прогноз log(RV) переводится через exp в RV;
QLIKE используется в форме `log(v_hat) + RV/v_hat`. Её значение может быть отрицательным;
сравнивать его можно на одинаковых наблюдениях.

Повторный запуск той же конфигурации переиспользует проверенный результат.
Для повторения с другой инициализацией меняйте seed. `RUN_TAG` помогает разделять серии опытов.

После каждого завершённого опыта обновляется `summary.csv` этой серии.
В `status.csv` виден результат каждого набора и seed; `errors.csv` содержит ошибки.
При ошибке следующий набор всё равно запускается; в конце частичная серия явно отмечается.
Повторный запуск переиспользует **завершённые** опыты с той же конфигурацией.
Прерванное обучение текущий runner не возобновляет: для него потребуется новый `RUN_TAG`.


In [4]:
batch = run_batch(
    DATA_ROOT, MODEL_KIND, RESULTS_ROOT,
    datasets=DATASETS, seeds=SEEDS, config=CONFIG,
    evaluate_test=EVALUATE_TEST, run_tag=RUN_TAG,
    inspect_dataset=inspect_dataset,
)
reports = batch["reports"]
summary = batch["summary"]
SUMMARY_DIR = batch["summary_dir"]
display(batch["status"])
display(summary)
print("Сводные результаты:", SUMMARY_DIR / "summary.csv")



=== lstm: synthetic ===


,dataset,split,samples,nodes,history,features_per_node
0,synthetic,train,4140,12,60,1
1,synthetic,val,600,12,60,1
2,synthetic,test,1201,12,60,1


split,test,train,val
node,,,
node_0,1201,4140,600
node_1,1201,4140,600
node_10,1201,4140,600
node_11,1201,4140,600
node_2,1201,4140,600
node_3,1201,4140,600
node_4,1201,4140,600
node_5,1201,4140,600
node_6,1201,4140,600


Один вход: (60, 12, 1) цель: (12,)
lstm seed=0 epoch=1: train MSE=0.98122; val mae=0.0884983
lstm seed=0 epoch=2: train MSE=0.90333; val mae=0.085689
lstm seed=0 epoch=3: train MSE=0.85415; val mae=0.0840348
lstm seed=0 epoch=4: train MSE=0.82584; val mae=0.0829193
lstm seed=0 epoch=5: train MSE=0.80961; val mae=0.0824393
lstm seed=0 epoch=6: train MSE=0.79894; val mae=0.0819153
lstm seed=0 epoch=7: train MSE=0.79167; val mae=0.0817238
lstm seed=0 epoch=8: train MSE=0.78626; val mae=0.0815159
lstm seed=0 epoch=9: train MSE=0.78214; val mae=0.0814047
lstm seed=0 epoch=10: train MSE=0.77916; val mae=0.0812848
lstm seed=0 epoch=11: train MSE=0.77643; val mae=0.0812639
lstm seed=0 epoch=12: train MSE=0.77453; val mae=0.0811707
lstm seed=0 epoch=13: train MSE=0.77277; val mae=0.0811095
lstm seed=0 epoch=14: train MSE=0.77159; val mae=0.0811005
lstm seed=0 epoch=15: train MSE=0.77015; val mae=0.0811155
Saved: /home/loktar/Desktop/money_coursework/results/synthetic/lstm/seed_000/9bbeecb1c88b


,dataset,split,samples,nodes,history,features_per_node
0,moex,train,870,10,60,1
1,moex,val,252,10,60,1
2,moex,test,254,10,60,1


split,test,train,val
node,,,
GAZP,254,867,252
GMKN,254,870,247
LKOH,254,870,252
MGNT,254,870,252
NVTK,254,870,252
PLZL,251,870,252
ROSN,254,870,252
SBER,254,870,252
VTBR,254,870,248


... (5 outputs left out)

Один вход: (60, 207, 1) цель: (207,)
lstm seed=0 epoch=1: train MSE=0.60842; val mae=4.09719
lstm seed=0 epoch=2: train MSE=0.47438; val mae=3.88664
lstm seed=0 epoch=3: train MSE=0.43664; val mae=3.72993
lstm seed=0 epoch=4: train MSE=0.41236; val mae=3.60958
lstm seed=0 epoch=5: train MSE=0.39831; val mae=3.59164
lstm seed=0 epoch=6: train MSE=0.38994; val mae=3.54481
lstm seed=0 epoch=7: train MSE=0.38453; val mae=3.48475
lstm seed=0 epoch=8: train MSE=0.38081; val mae=3.47744
lstm seed=0 epoch=9: train MSE=0.37843; val mae=3.46905
lstm seed=0 epoch=10: train MSE=0.37667; val mae=3.45617
lstm seed=0 epoch=11: train MSE=0.37544; val mae=3.45586
lstm seed=0 epoch=12: train MSE=0.37426; val mae=3.43703
lstm seed=0 epoch=13: train MSE=0.37346; val mae=3.48519
lstm seed=0 epoch=14: train MSE=0.37296; val mae=3.43515
lstm seed=0 epoch=15: train MSE=0.37237; val mae=3.43817
Saved: /home/loktar/Desktop/money_coursework/results/metr_la/lstm/seed_000/67a2d4fb4bac
Batch summaries: /home/loktar

,dataset,model,seed,status,error,output_dir
0,synthetic,lstm,0,completed,,/home/loktar/Desktop/money_coursework/results/...
1,moex,lstm,0,completed,,/home/loktar/Desktop/money_coursework/results/...
2,metr_la,lstm,0,completed,,/home/loktar/Desktop/money_coursework/results/...


,dataset,model,seed,split,primary_metric,primary_score,mae,rmse,qlike,normalized_mse,n_observed,n_origins,best_epoch,parameters,fit_seconds,comparison_id,run_id,output_dir
0,synthetic,lstm,0,val,mae,0.081100,0.081100,0.101304,NaN,0.791871,7200,600,14,2124,11.641013,36989cce73a0,9bbeecb1c88b,/home/loktar/Desktop/money_coursework/results/...
1,synthetic,lstm,0,test,mae,0.080639,0.080639,0.101053,NaN,0.787945,14412,1201,14,2124,11.641013,36989cce73a0,9bbeecb1c88b,/home/loktar/Desktop/money_coursework/results/...
2,moex,lstm,0,val,qlike,-7.398339,0.000145,0.000304,-7.398339,0.642902,2371,252,15,1962,2.750675,362a12d77352,d887a9ea5ac7,/home/loktar/Desktop/money_coursework/results/...
3,moex,lstm,0,test,qlike,-7.227486,0.000155,0.000589,-7.227486,0.585428,2283,254,15,1962,2.750675,362a12d77352,d887a9ea5ac7,/home/loktar/Desktop/money_coursework/results/...
4,metr_la,lstm,0,val,mae,3.435151,3.435151,5.788873,NaN,0.407770,660000,3246,14,17919,163.245416,c43da30bee45,67a2d4fb4bac,/home/loktar/Desktop/money_coursework/results/...
5,metr_la,lstm,0,test,mae,3.675398,3.675398,5.974551,NaN,0.405432,1247252,6189,14,17919,163.245416,c43da30bee45,67a2d4fb4bac,/home/loktar/Desktop/money_coursework/results/...


Сводные результаты: /home/loktar/Desktop/money_coursework/results/summaries/lstm/20261124T081908_197252Z/summary.csv


## Сохранённые результаты и примеры

Подробные результаты: `results/<dataset>/<model>/seed_<seed>/<run_id>/`.
Там находятся `run.json`, метрики, ошибки по датам `*_losses.csv`, метрики по вершинам,
прогнозы с исходной маской в `*_predictions.npz`. У нейросетей также сохраняются история обучения и лучшая модель.

Сводка этой серии: `results/summaries/<model>/<дата_UTC>/summary.csv`.
Одна строка — один набор, seed и разбиение val/test; при одном seed и трёх успешных наборах получится 6 строк.
В сводке есть основная метрика, MAE, RMSE, QLIKE для MOEX, число целей, время обучения и путь к опыту.
Разные серии получают отдельные папки. Ниже показаны примеры для каждого успешно обработанного набора.


In [5]:
preview_split = "test" if EVALUATE_TEST else "val"
for dataset_name in DATASETS:
    dataset_reports = [report for report in reports if report["dataset"] == dataset_name]
    if not dataset_reports:
        continue
    run_dir = Path(dataset_reports[0]["output_dir"])
    print(dataset_name, "—", run_dir)
    display(pd.read_csv(run_dir / f"{preview_split}_per_node.csv").head(12))
    with np.load(run_dir / f"{preview_split}_predictions.npz") as saved:
        preview = pd.DataFrame({
            "node": saved["node_order"], "observed": saved["mask"][0],
            "actual_raw": saved["target_raw"][0], "prediction_raw": saved["prediction_raw"][0],
        })
    display(preview.head(12))


synthetic — /home/loktar/Desktop/money_coursework/results/synthetic/lstm/seed_000/9bbeecb1c88b


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,node_0,node_0,1201,1201,0.081767,0.102864
1,1,node_1,node_1,1201,1201,0.077041,0.096577
2,2,node_2,node_2,1201,1201,0.081733,0.102688
3,3,node_3,node_3,1201,1201,0.080218,0.098624
4,4,node_4,node_4,1201,1201,0.077968,0.098634
5,5,node_5,node_5,1201,1201,0.079087,0.099839
6,6,node_6,node_6,1201,1201,0.080979,0.102767
7,7,node_7,node_7,1201,1201,0.081433,0.100206
8,8,node_8,node_8,1201,1201,0.079734,0.100930
9,9,node_9,node_9,1201,1201,0.083868,0.103598


,node,observed,actual_raw,prediction_raw
0,node_0,True,-0.018677,-0.048100
1,node_1,True,0.002347,0.015418
2,node_2,True,-0.135561,-0.001805
3,node_3,True,0.063401,0.028008
4,node_4,True,0.048733,0.010168
5,node_5,True,-0.123078,-0.067460
6,node_6,True,-0.013832,-0.056718
7,node_7,True,-0.040185,-0.020335
8,node_8,True,0.174519,0.024748
9,node_9,True,-0.034389,-0.002275


moex — /home/loktar/Desktop/money_coursework/results/moex/lstm/seed_000/d887a9ea5ac7


,node_id,node,name,observed_pairs,n_observed,mae,rmse,qlike
0,0,SBER,SBER,254,254,0.000091,0.000147,-8.021614
1,1,GAZP,GAZP,254,254,0.000173,0.000347,-6.897077
2,2,LKOH,LKOH,254,254,0.000131,0.000222,-7.613006
3,3,GMKN,GMKN,254,254,0.000138,0.000223,-7.035820
4,4,YNDX,YNDX,0,0,NaN,NaN,NaN
5,5,ROSN,ROSN,254,254,0.000116,0.000187,-7.409092
6,6,PLZL,PLZL,251,251,0.000154,0.000276,-7.183495
7,7,MGNT,MGNT,254,254,0.000143,0.000268,-7.249265
8,8,NVTK,NVTK,254,254,0.000190,0.000352,-6.822652
9,9,VTBR,VTBR,254,254,0.000262,0.001604,-6.814834


,node,observed,actual_raw,prediction_raw
0,SBER,True,0.000196,0.000188
1,GAZP,True,0.000641,0.000204
2,LKOH,True,0.000351,0.000238
3,GMKN,True,0.000510,0.000208
4,YNDX,False,NaN,0.000355
5,ROSN,True,0.000478,0.000253
6,PLZL,True,0.000058,0.000251
7,MGNT,True,0.000848,0.000214
8,NVTK,True,0.000861,0.000314
9,VTBR,True,0.000401,0.000333


metr_la — /home/loktar/Desktop/money_coursework/results/metr_la/lstm/seed_000/67a2d4fb4bac


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,773869,773869,6188,6188,3.145888,5.517149
1,1,767541,767541,6189,6189,1.711527,2.744137
2,2,767542,767542,6189,6189,2.795258,4.604338
3,3,717447,717447,6185,6185,3.171603,4.586376
4,4,717446,717446,5963,5963,4.415096,6.145028
5,5,717445,717445,6185,6185,5.868569,7.799113
6,6,773062,773062,6185,6185,3.983009,6.399930
7,7,767620,767620,6186,6186,1.638511,2.334774
8,8,737529,737529,6088,6088,3.340652,5.195552
9,9,717816,717816,6186,6186,6.446784,9.627397


,node,observed,actual_raw,prediction_raw
0,773869,True,64.750,65.005132
1,767541,True,56.375,61.257345
2,767542,True,69.375,66.809788
3,717447,True,58.000,57.370452
4,717446,True,66.000,66.989153
5,717445,True,44.375,59.406411
6,773062,True,63.125,65.952557
7,767620,True,52.250,62.180279
8,737529,True,65.750,61.264215
9,717816,True,68.000,61.128468


## Сопоставление четырёх запусков

Сравнивайте строки с одним `comparison_id`: он фиксирует данные, нормировку, общие настройки и версии кода.
Метрики разных наборов данных имеют разные единицы и напрямую не сопоставляются.

Обычная LSTM здесь многомерная, а LSTM–GNN использует общий по вершинам временной кодировщик.
Поэтому их сравнение относится к двум семействам моделей, а не к изолированному добавлению одного слоя.
Главная проверка обновления графа — **static против dynamic** при одинаковых seed и настройках.

Текущая синтетика со статическим истинным графом служит контрольным сценарием.
Для проверки пользы изменения топологии позже потребуется синтетика с меняющимися связями.

`comparison.csv` собирает уже завершённые запуски других моделей с теми же протоколом и seed
внутри каждого набора. После запуска всех четырёх ноутбуков таблица последнего включает все четыре модели.
При одном seed и наличии val/test это 24 строки. Ошибки разных наборов не усредняются.


In [6]:
comparison_parts = []
for dataset_name in DATASETS:
    current_ids = summary.loc[summary["dataset"].eq(dataset_name), "comparison_id"].unique()
    for split in (("val", "test") if EVALUATE_TEST else ("val",)):
        table = collect_results(RESULTS_ROOT, dataset_name, split=split, run_tag=RUN_TAG)
        if table.empty:
            continue
        table = table.loc[table["comparison_id"].isin(current_ids) & table["seed"].isin(SEEDS)].copy()
        if not table.empty:
            table.insert(0, "dataset", dataset_name)
            comparison_parts.append(table)
comparison = pd.concat(comparison_parts, ignore_index=True) if comparison_parts else pd.DataFrame(
    columns=["dataset", "model", "seed", "split", "comparison_id", "mae", "rmse", "qlike"]
)
comparison.to_csv(SUMMARY_DIR / "comparison.csv", index=False)
display(comparison)
print("Сопоставимые запуски:", SUMMARY_DIR / "comparison.csv")

failed = batch["status"].loc[batch["status"]["status"].ne("completed")]
if not failed.empty:
    display(failed)
    raise RuntimeError(f"Серия завершена частично. Ошибки: {SUMMARY_DIR / 'errors.csv'}")
print("Готово: результаты сохранены для всех трёх наборов.")


,dataset,model,seed,split,comparison_id,mae,rmse,n_observed,best_epoch,parameters,fit_seconds,output_dir,qlike
0,synthetic,lstm,0,val,36989cce73a0,0.081100,0.101304,7200,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
1,synthetic,lstm_gnn_static,0,val,36989cce73a0,0.081928,0.102395,7200,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
2,synthetic,persistence,0,val,36989cce73a0,0.096973,0.121423,7200,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
3,synthetic,lstm,0,test,36989cce73a0,0.080639,0.101053,14412,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
4,synthetic,lstm_gnn_static,0,test,36989cce73a0,0.081709,0.102267,14412,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
5,synthetic,persistence,0,test,36989cce73a0,0.096982,0.121153,14412,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
6,moex,lstm,0,val,362a12d77352,0.000145,0.000304,2371,15.0,1962,2.750675,/home/loktar/Desktop/money_coursework/results/...,-7.398339
7,moex,lstm_gnn_static,0,val,362a12d77352,0.000134,0.000290,2371,14.0,1761,6.221492,/home/loktar/Desktop/money_coursework/results/...,-7.465629
8,moex,persistence,0,val,362a12d77352,0.000167,0.000364,2371,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,-7.368911
9,moex,lstm,0,test,362a12d77352,0.000155,0.000589,2283,15.0,1962,2.750675,/home/loktar/Desktop/money_coursework/results/...,-7.227486


Сопоставимые запуски: /home/loktar/Desktop/money_coursework/results/summaries/lstm/20261124T081908_197252Z/comparison.csv
Готово: результаты сохранены для всех трёх наборов.
